# ReproCheckFlow

Runs the project in isolated blocks for debugging, instead of a full `kickoff()`.

- **Section A** exercises `ReproCheckerCrew` directly (crews and individual tasks).
- **Section B** exercises `ReproCheckFlow` (from `main.py`) one `@start`/`@listen`/`@router` step at a time.

Re-run cells individually as needed; each section is self-contained after Setup runs once.

> Note: `filter_paper()` and `run_repro_check()` write results back into the CSV at `INPUTS_PATH`
> (same side effect as the real flow). Re-running them repeatedly will overwrite that row again.


## Setup

Imports, config, and shared debug variables.

In [2]:
import json
from pathlib import Path

import pandas as pd

from flow_reproassesslsm_st1.config import INPUTS_PATH, OUTPUT_DIR, PDF_DIR
from flow_reproassesslsm_st1.models import (
    ReproCheckState,
    FilterOutput,
    DatasetEntry,
    DataReproOutput,
    MethodEntry,
    MethodReproOutput,
    AvailabilityOutput,
    ReproducibilityAssessment,
    ReproducibilityReport,
)
from flow_reproassesslsm_st1.crews.reprochecker_crew.reprochecker_crew import ReproCheckerCrew
from flow_reproassesslsm_st1.main import ReproCheckFlow

print(f"INPUTS_PATH = {INPUTS_PATH.resolve()}")
print(f"PDF_DIR     = {PDF_DIR.resolve()}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR.resolve()}")


INPUTS_PATH = D:\Flávio Rocha USER\OneDrive\University of Twente\MSc 2025-2027\Thesis_LAReprod\LSM_ReproChecker\publications\scopus_export_Jul_22_2026_query1.csv
PDF_DIR     = D:\Flávio Rocha USER\OneDrive\University of Twente\MSc 2025-2027\Thesis_LAReprod\LSM_ReproChecker\publications
OUTPUT_DIR  = D:\Flávio Rocha USER\OneDrive\University of Twente\MSc 2025-2027\Thesis_LAReprod\LSM_ReproChecker\output2


### Debug inputs

One row's worth of data, used by both sections below. Defaults to the Lindsay et al. 2022
paper (already present in `publications/`). Swap these for any other row in `INPUTS_PATH`.

Alternatively, pull a row (and its abstract) straight from the CSV by EID:

In [ ]:
# "2-s2.0-85130393221" = Lindsay E. et al 2022 / Multi-Temporal Satellite Image Composites in Google Earth Engine for Improved Landslide Visibility: A Case Study of a Glacial Landscape
# "2-s2.0-85087214140" = Mohan A et al 2021 / Review on remote sensing methods for landslide detection using machine and deep learning / 

publication_id = "2-s2.0-85087214140"
pdf_file = publication_id + ".pdf"

print((PDF_DIR / pdf_file).resolve(), (PDF_DIR / pdf_file).exists())

df = pd.read_csv(INPUTS_PATH, sep=";")
df.columns = df.columns.str.strip()

row = df.loc[df["EID"].astype(str).str.strip() == publication_id].squeeze()

doi_url = f"https://doi.org/{str(row['DOI']).strip()}"
abstract = str(row["Abstract"]).strip()

row

Authors                              Mohan A.; Singh A.K.; Kumar B.; Dwivedi R.
Author full names             Mohan, Amrita (57196078933); Singh, Amit Kumar...
Author(s) ID                  57196078933; 55726466900; 57209952119; 5719881...
Title                         Review on remote sensing methods for landslide...
Year                                                                       2021
Source title                  Transactions on Emerging Telecommunications Te...
Volume                                                                     32.0
Issue                                                                         7
Art. No.                                                                  e3998
Page start                                                                  NaN
Page end                                                                    NaN
Cited by                                                                    317
DOI                                     

---
## Section A — `ReproCheckerCrew` (crews/reprochecker_crew/reprochecker_crew.py)

Debug the crew in isolation, without going through the Flow's state management.

In [8]:
crew = ReproCheckerCrew(pdf_file=pdf_file)
crew

### A.1 — `filter_crew()` (abstract-only screening)

In [9]:
filter_result = await crew.filter_crew().kickoff_async(inputs={"abstract": abstract})
filter_result.pydantic

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.4                                                                                        │
│  Latest version:  1.15.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: ReproCheckerCrew                                                                                         │
│  ID: 73d08f04-afe9-4d6e-b5c9-4f07de281325                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: filter_landslide_mapping_paper                                                                           │
│  ID: f57e1bde-94fd-4c1f-808a-cc3655eeeabc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Landslide Mapping Abstract Screener                                                                     │
│                                                                                                                 │
│  Task: Based only on the abstract provided below, determine whether this paper explores and documents a         │
│  landslide mapping method. A qualifying paper's abstract must indicate (or strongly imply) a methodology (or a  │
│  reference to prior methodology), a description of the datasets used, and a description of the results.         │
│  Exclude papers that only describe landslide inventories, landslide susceptibility mapping methods, or that     │
│  are review articles. Do not assume details the abstract doesn't state; if genuinely ambiguous, lean toward     │
│  EXCLUDE and say why in the reason.                                                                             │
│  Abstract: Landslide, one of the most critical natural hazards, is caused due to specific compositional slope   │
│  movement. In the past decades, due to inflation of urbanized area and climate change, a compelling expansion   │
│  in landslide prevalence took place which is also termed as mass/slope movement and mass wasting, causing       │
│  extensive collapse around the world. The principal reason for its pursuance is a reduction in the internal     │
│  resistance of soil and rocks, classified as a slide, topple, fall, and flow. Slopes can be differentiated      │
│  based on earth material and the nature of its movements. The downward flow of landslides occurs due to         │
│  excessive rainfall, snowmelt, earthquake, volcanic eruption, and so on. This review article revisits the       │
│  conventional approaches for identification of landslides, predicting future risk, associated with slope        │
│  failures, followed by emphasizing the advantages of modern geospatial techniques such as aerial                │
│  photogrammetry, satellite remote sensing images (ie, panchromatic, multispectral, radar images), Terrestrial   │
│  laser scanning, and High-Resolution Digital Elevation Model (HR-DEM) in updating landslide inventory maps.     │
│  Machine learning techniques like Support Vector Machine, Artificial neural network, deep learning has been     │
│  extensively used with geographical data producing effective results for assessment of natural                  │
│  hazard/resources and environmental research. Based on recent studies, deep learning is a reliable tool         │
│  addressing remote sensing challenges such as trade-off in imaging system producing poor quality                │
│  investigation, in addition, to expedite consequent task such as image recognition, object detection,           │
│  classification, and so on. Conventional methods, like pixel and object-based machine learning methods, have    │
│  been broadly explored. Advanced development in deep learning technique like CNN (Convolutional neural          │
│  network) has been extensively successful in information extraction from an image and has exceeded other        │
│  traditional approaches. Over the past few years, minor attempts have been made for landslide susceptibility    │
│  mapping using CNN. In addition, small sample sizes for training purpose will be major drawback and notably     │
│  remarkable while using deep learning techniques. Also, assessment of the model's performance with diverse      │
│  training and testing proportion other than commonly utilized ratio, that is, 70/30 needs to be explored        │
│  further. The review article briefly highlights the remote sensing methods for landslide detection using        │
│  machine learning and deep learning. © 2020 John Wiley 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Landslide Mapping Abstract Screener                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  decision='EXCLUDE' reason='review article only, missing explicit description of a landslide mapping method,    │
│  dataset, and results'                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: filter_landslide_mapping_paper                                                                           │
│  Agent: Landslide Mapping Abstract Screener                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: ReproCheckerCrew                                                                                         │
│  ID: 73d08f04-afe9-4d6e-b5c9-4f07de281325                                                                       │
│  Final Output: {"decision":"EXCLUDE","reason":"review article only, missing explicit description of a           │
│  landslide mapping method, dataset, and results"}                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FilterOutput(decision='EXCLUDE', reason='review article only, missing explicit description of a landslide mapping method, dataset, and results')

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### A.2 — `repro_crew()` (full reproducibility checks)

Only meaningful if the paper would pass the filter (`decision == "INCLUDE"`).
Kicks off `check_data_reproducibility`, `check_method_reproducibility`,
`check_artifact_availability`, and `compile_final_report` in sequence.

In [10]:
repro_result = await crew.repro_crew().kickoff_async(inputs={"doi_url": doi_url})
repro_result.pydantic

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.4                                                                                        │
│  Latest version:  1.15.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: ReproCheckerCrew                                                                                         │
│  ID: cbfc7e0d-f2b9-4c01-bb07-0cbe62b64ac9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: check_data_reproducibility                                                                               │
│  ID: 9b71467d-32ba-4054-843d-fe956dbeb2d1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Landslide Mapping Paper Analyst                                                                         │
│                                                                                                                 │
│  Task: Use your PDF search tool to identify all datasets used in the paper, including satellite imagery and     │
│  landslide inventories. Search for data sources, dataset names, and any retrieval links or access statements.   │
│  For each dataset, determine its source and whether the paper provides a direct link or clear indication of     │
│  where to retrieve it.                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'satellite imagery dataset used in this study'}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Reproducibility Report Elaborator                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  reproducibility_assessment='Partially reproducible: datasets are named with retrieval links, but the custom    │
│  classification code is not available and no repository is provided.'                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: compile_final_report                                                                                     │
│  Agent: Reproducibility Report Elaborator                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: ReproCheckerCrew                                                                                         │
│  ID: cbfc7e0d-f2b9-4c01-bb07-0cbe62b64ac9                                                                       │
│  Final Output: {"reproducibility_assessment":"Partially reproducible: datasets are named with retrieval links,  │
│  but the custom classification code is not available and no repository is provided."}                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ReproducibilityAssessment(reproducibility_assessment='Partially reproducible: datasets are named with retrieval links, but the custom classification code is not available and no repository is provided.')

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### A.3 — Individual task outputs

Each `@task` method is memoized by `CrewBase`, so calling it again after the crew
kickoff returns the same `Task` instance — now populated with `.output`.

In [11]:
data_output = crew.check_data_reproducibility().output.pydantic       # DataReproOutput
method_output = crew.check_method_reproducibility().output.pydantic   # MethodReproOutput
avail_output = crew.check_artifact_availability().output.pydantic     # AvailabilityOutput
assessment_output = crew.compile_final_report().output.pydantic       # ReproducibilityAssessment

data_output

DataReproOutput(datasets=[DatasetEntry(name='Sentinel-2 Level 2A imagery (bottom of atmosphere reflectance, used for dNDVI bi-temporal and multi-temporal change detection)', source='European Space Agency (ESA) Copernicus programme', link='https://scihub.copernicus.eu/dhus/#/home', status='AVAILABLE'), DatasetEntry(name='Sentinel-1 SAR imagery (Level 1 SLC and GRD, dual-polarised VV/VH, IW mode, used for bi-temporal and multi-temporal SAR-RGB composites)', source='European Space Agency (ESA) Copernicus programme', link='https://scihub.copernicus.eu/dhus/#/home', status='AVAILABLE'), DatasetEntry(name='Sentinel-1 and Sentinel-2 imagery accessed directly via Google Earth Engine (GEE) code editor', source='Google Earth Engine / ESA Copernicus', link='https://code.earthengine.google.com/', status='AVAILABLE'), DatasetEntry(name='Digital Terrain Model (DTM), 10 m resolution, used for terrain correction of SAR imagery (tiles 6800_1 to 6800_4)', source='Høydedata (Norwegian national height/ter

In [12]:
method_output

MethodReproOutput(methods=[MethodEntry(name='Manual digitization of landslides from Sentinel-2 Bi-Temporal (S2-BT) dNDVI composite', method_type='MANUAL', summary='Preliminary landslide mapping was conducted using the conventional S2-BT approach, completed within a week of the landslide occurrence (~3 hours). Pre- and post-event Level 2A Sentinel-2 tiles were processed in SNAP 7.0 software (spatial subset, collocation, band math) to calculate a difference NDVI (dNDVI) raster: dNDVI = (post NIR-Red)/(post NIR+Red) - (pre NIR-Red)/(pre NIR+Red). Polygons delimiting assumed landslides (dark elongated pixel clusters) were then drawn manually in SNAP. This preliminary map was later verified/updated through field visits, a helicopter flight, and comparison with a detailed field-based inventory produced by co-author D. Rüther, and used as the baseline for comparing three other change-detection approaches.', code_status='N/A', code_link=None, reused_citation=None), MethodEntry(name='Manual lan

In [13]:
avail_output

AvailabilityOutput(access_status='NOT_ACCESSIBLE', data_status='NOT_STATED', data_links=[], code_status='NOT_STATED', code_links=[], author_statement='The publication page could not be accessed (HTTP 403 error at https://onlinelibrary.wiley.com/doi/10.1002/ett.3998), so no data or code availability statements could be retrieved or verified.')

In [14]:
assessment_output

ReproducibilityAssessment(reproducibility_assessment='Partially reproducible: datasets are named with retrieval links, but the custom classification code is not available and no repository is provided.')

### A.4 — Assemble the final `ReproducibilityReport`

In [15]:
final_report = ReproducibilityReport(
    datasets=data_output.datasets,
    methods=method_output.methods,
    availability=avail_output,
    reproducibility_assessment=assessment_output.reproducibility_assessment,
)

In [16]:
df = pd.read_csv(INPUTS_PATH, sep=';')
df.columns = df.columns.str.strip()

mask = df["EID"].astype(str).str.strip() == str(publication_id).strip()
if not mask.any():
    print(f"Warning: EID {publication_id} not found in {INPUTS_PATH}, findings not recorded.")

avail = final_report.availability

df.loc[mask, "Datasets"] = json.dumps([d.model_dump() for d in final_report.datasets])
df.loc[mask, "Methods"] = json.dumps([m.model_dump() for m in final_report.methods])

df.loc[mask, "Webpage_Access_Status"] = avail.access_status
df.loc[mask, "Webpage_Data_Status"] = avail.data_status
df.loc[mask, "Webpage_Data_Links"] = ", ".join(avail.data_links) if avail.data_links else ""
df.loc[mask, "Webpage_Code_Status"] = avail.code_status
df.loc[mask, "Webpage_Code_Links"] = ", ".join(avail.code_links) if avail.code_links else ""
df.loc[mask, "Webpage_Author_Statement"] = avail.author_statement or ""
df.loc[mask, "Reproducibility_Assessment"] = final_report.reproducibility_assessment

df.to_csv(INPUTS_PATH, sep=';', index=False)

TypeError: Invalid value '' for dtype 'float64'

In [17]:
final_report = final_report.model_dump_json(indent=2)
try:
    print("Saving report")
    OUTPUT_DIR.mkdir(exist_ok=True)
    output_file = OUTPUT_DIR / f"{publication_id}_repro_report.json"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(final_report)
    print(f"Report saved to {output_file}")
except Exception as e:
    print(f"Error saving report: {e}.\nFinal report:\n{final_report}")

---
## Section B — `ReproCheckFlow` (main.py)

The `@start`/`@listen`/`@router` decorators just tag the method for the Flow engine's
own `kickoff()` orchestration — the underlying methods stay directly callable on an
instance, so each step below runs it standalone against `flow.state`.

In [4]:
flow = ReproCheckFlow()
flow.state

StateWithId(publication_id='', pdf_file='', doi='', abstract='', filter_decision='', filter_reason='', final_report=None, id='eb072a36-72c1-4407-806f-642b7a06f676')

### B.1 — `load_inputs` (`@start`)

Sets `flow.state` from a trigger payload and instantiates `flow._crew`.

In [5]:
flow.load_inputs(crewai_trigger_payload={
    "publication_id": publication_id,
    "pdf_file": pdf_file,
    "doi": doi,
    "abstract": abstract,
})
flow.state

Loading inputs
Using trigger payload: {'publication_id': '2-s2.0-85130393221', 'pdf_file': '2-s2.0-85130393221.pdf', 'doi': '10.3390/rs14102301', 'abstract': 'Regional early warning systems for landslides rely on historic data to forecast future events and to verify and improve alarms. However, databases of landslide events are often spatially biased towards roads or other infrastructure, with few reported in remote areas. In this study, we demonstrate how Google Earth Engine can be used to create multi-temporal change detection image composites with freely available Sentinel-1 and-2 satellite images, in order to improve landslide visibility and facilitate landslide detection.'}
PDF pdf_file: 2-s2.0-85130393221.pdf
DOI: 10.3390/rs14102301


InternalServerError: Error code: 503 - {'error': 'Too many concurrent requests'} in upsert.

### B.2 — `filter_paper` (`@listen(load_inputs)`)

Runs `filter_crew()` and writes `Filter_Decision`/`Filter_Reason` back into `INPUTS_PATH`.

In [ ]:
decision = flow.filter_paper()
print("decision:", decision)
print("reason:", flow.state.filter_reason)

### B.3 — `route_on_filter` (`@router(filter_paper)`)

In [ ]:
route = flow.route_on_filter(decision)
print("route:", route)

### B.4a — `write_exclusion` (`@listen("excluded")`)

Only run this cell if `route == "excluded"`.

In [ ]:
assert route == "excluded", f"route was {route!r}, not 'excluded'"
flow.write_exclusion()
flow.state.final_report

### B.4b — `run_repro_check` (`@listen("included")`)

Only run this cell if `route == "included"`. Runs `repro_crew()` and writes the
datasets/methods/availability/assessment columns back into `INPUTS_PATH`.

In [ ]:
assert route == "included", f"route was {route!r}, not 'included'"
flow.run_repro_check()
flow.state.final_report

### B.5 — `save_report` (`@listen(run_repro_check)`)

Writes `flow.state.final_report` to `OUTPUT_DIR/{publication_id}_repro_report.json`.
Note: in the real flow this only fires after `run_repro_check`, but it just reads
`flow.state.final_report`, so it works after either B.4a or B.4b for debugging.

In [ ]:
flow.save_report()

---
## Full flow kickoff (end-to-end, for comparison)

Runs everything above through the Flow engine's own orchestration in one call.

In [ ]:
# full_flow = ReproCheckFlow()
# result = full_flow.kickoff(inputs={
#     "publication_id": publication_id,
#     "pdf_file": pdf_file,
#     "doi": doi,
#     "abstract": abstract,
# })
# full_flow.state.final_report